In [ ]:

#####################################################################
# STEP 1: IMPORTING LIBRARIES & HARDWARE CHECK
#####################################################################
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
os.environ['NCCL_DEBUG'] = 'WARN'
print("[INFO] Loading required python libraries...")
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from glob import glob
from PIL import Image

import tensorflow as tf
print("\n" + "="*50)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("Name: ", tf.config.list_physical_devices('GPU'))
print("="*50 + "\n")

from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Recall, Precision
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, LeakyReLU, Add
from tensorflow.keras.models import Model

#####################################################################
# STEP 2: PROJECT CONSTANTS
#####################################################################
H = 256
W = 256
BATCH_SIZE_PER_REPLICA = 8 
LEARNING_RATE = 1e-4
EPOCHS = 150  
LR_PATIENCE = 5
ES_PATIENCE = 10
SMOOTH = 1e-6

#####################################################################
# STEP 3: DATA PREPROCESSING PIPELINE (CLEAN 80/10/10 SPLIT)
#####################################################################
def load_data(data_path):
    print(f"[INFO] Scanning for data inside: {data_path} ...")
    if not os.path.exists(data_path):
        print(f"[ERROR] CRITICAL FAILURE: The directory {data_path} does not exist!")
        return ([], []), ([], []), ([], [])

    images = sorted(glob(os.path.join(data_path, "*_sat.jpg")))
    masks = sorted(glob(os.path.join(data_path, "*_mask.png")))
    
    if len(images) == 0:
        print("[ERROR] No files matching pattern found.")
        return ([], []), ([], []), ([], [])

    # Clean 80/10/10 split
    train_x, temp_x, train_y, temp_y = train_test_split(images, masks, test_size=0.2, random_state=42)
    val_x, test_x, val_y, test_y = train_test_split(temp_x, temp_y, test_size=0.5, random_state=42)
    
    return (train_x, train_y), (val_x, val_y), (test_x, test_y)

def read_image(path):
    try:
        img = Image.open(path).convert('RGB').resize((W, H))
        return np.array(img, dtype=np.float32) / 255.0
    except: return None

def read_mask(path):
    try:
        mask = Image.open(path).convert('L').resize((W, H))
        return np.expand_dims(np.array(mask, dtype=np.float32) / 255.0, axis=-1)
    except: return None

def tf_parse(x, y):
    def _parse(x, y): return read_image(x), read_mask(y)
    x, y = tf.numpy_function(_parse, [x, y], [tf.float32, tf.float32])
    x.set_shape([H, W, 3]); y.set_shape([H, W, 1])
    return x, y

#####################################################################
# STEP 4: MODEL ARCHITECTURE (RES-UNET 34)
#####################################################################
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    return x

def residual_block(x, filters):
    res = Conv2D(filters, (1, 1), padding='same')(x)
    res = BatchNormalization()(res)
    x = conv_block(x, filters)
    x = Add()([x, res])
    x = LeakyReLU(negative_slope=0.1)(x)
    return x

def build_resnet(input_shape=(256, 256, 3)):
    inputs = Input(input_shape)
    c1 = residual_block(inputs, 64); p1 = MaxPool2D((2, 2))(c1)
    c2 = residual_block(p1, 128); p2 = MaxPool2D((2, 2))(c2)
    c3 = residual_block(p2, 256); p3 = MaxPool2D((2, 2))(c3)
    c4 = residual_block(p3, 512); p4 = MaxPool2D((2, 2))(c4)
    
    bn = residual_block(p4, 1024)
    
    d1 = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(bn)
    d1 = Concatenate()([d1, c4]); d1 = residual_block(d1, 512)
    d2 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(d1)
    d2 = Concatenate()([d2, c3]); d2 = residual_block(d2, 256)
    d3 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(d2)
    d3 = Concatenate()([d3, c2]); d3 = residual_block(d3, 128)
    d4 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(d3)
    d4 = Concatenate()([d4, c1]); d4 = residual_block(d4, 64)
    
    outputs = Conv2D(1, (1, 1), padding='same', activation='sigmoid')(d4)
    return Model(inputs, outputs)

#####################################################################
# STEP 5: TRAINING THE MODEL (KAGGLE DUAL GPU)
#####################################################################
DATASETS = [
    {"name": "DeepGlobe", "path": "/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/train"},
]

for ds in DATASETS:
    print(f"\n{'='*50}\n TRAINING CYCLE: {ds['name']}\n{'='*50}")
    
    # ---------------------------------------------------------
    # 1. LOAD DATA
    # ---------------------------------------------------------
    (train_x, train_y), (val_x, val_y), (test_x, test_y) = load_data(ds['path'])
    if not train_x: continue

    # ---------------------------------------------------------
    # 2. CONSTRUCT TF_DATASET (DUAL-GPU TARGETING)
    # ---------------------------------------------------------
    strategy = tf.distribute.MirroredStrategy()
    GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
    
    print(f"[INFO] Using Global Batch Size: {GLOBAL_BATCH_SIZE}")
    train_dataset = tf.data.Dataset.from_tensor_slices((train_x, train_y)).map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
    if False:
        print("[INFO] Data Augmentation is ENABLED.")
        train_dataset = train_dataset.map(lambda x, y: (x, y), num_parallel_calls=tf.data.AUTOTUNE)
    
    train_dataset = train_dataset.shuffle(buffer_size=500).repeat().batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_dataset = tf.data.Dataset.from_tensor_slices((val_x, val_y)).map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE).batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
    train_dataset = train_dataset.with_options(options)
    val_dataset = val_dataset.with_options(options)

    # ---------------------------------------------------------
    # 3. BUILD & COMPILE MODEL WITH GPU MIRRORED SCOPE
    # ---------------------------------------------------------
    with strategy.scope():
        model = build_resnet(input_shape=(256, 256, 3))

        def iou(y_true, y_pred):
            y_pred = tf.cast(y_pred > 0.5, tf.float32)
            intersection = tf.reduce_sum(y_true * y_pred)
            union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
            return (intersection + SMOOTH) / (union + SMOOTH)

        print("[INFO] Compiling model inside GPU mapping strategy...")
        model.compile(loss='binary_crossentropy', optimizer=Adam(LEARNING_RATE), metrics=[iou, Precision(), Recall()])
    
    model.summary()
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_iou', mode='max', patience=ES_PATIENCE, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_iou', factor=0.5, patience=LR_PATIENCE, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath=f"best_model_resnet34_baseline.keras", monitor='val_iou', mode='max', save_best_only=True, verbose=1)
    ]

    # ---------------------------------------------------------
    # 4. TRAIN THE MODEL
    # ---------------------------------------------------------
    print("[INFO] Starting training loop...")
    history = model.fit(train_dataset, epochs=EPOCHS, steps_per_epoch=np.ceil(len(train_x)/GLOBAL_BATCH_SIZE).astype(int), validation_data=val_dataset, callbacks=callbacks)
    
    # ---------------------------------------------------------
    # 5. SAVE WEIGHTS
    # ---------------------------------------------------------
    model.save(f"final_model_resnet34_baseline.keras")
    print(f"[SUCCESS] Model saved to final_model_resnet34_baseline.keras")

    # ---------------------------------------------------------
    # 6. PLOT LOSS CURVES
    # ---------------------------------------------------------
    print("[INFO] Plotting performance metrics...")
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Val'); plt.title('Loss Curve'); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(history.history['iou'], label='Train'); plt.plot(history.history['val_iou'], label='Val'); plt.title('IoU Curve'); plt.legend()
    plt.show()
    

